# Opacity Study (update: June 7 by Shano)

Version 1.7

Set up the named colors for the study, and added functions to allow the color palettes to be randomly assigned to the left chart/right chart while fixing the order of colors on the three layers of the curves.

In [1]:
import revisitpy as rvt
import numpy as np
import pandas as pd
import altair as alt
import vl_convert as vlc
import itertools
import revisitpy_server as rs
import json
import time
import anywidget
import vega



# Meta Data
study_metadata = rvt.studyMetadata(
    authors=["Shano Liang"],
    organizations=["VIS Lab"],
    title='Opacity Judgment Study',
    description='',
    date='2025-06-07',
    version='1.7'
)


# UI Config
ui_config = rvt.uiConfig(
  contactEmail="sliang1@wpi.edu",
  logoPath="./assets/revisitLogoSquare.svg",
  sidebar=True,
  withProgressBar=False,
  nextOnEnter=True
)

# Introduction
introduction = rvt.component(type='markdown', path='./assets/introduction.md', component_name__= 'introduction')

# Snippet of the introduction component.
print(introduction)

{
    "correctAnswer": [],
    "path": "./assets/introduction.md",
    "response": [],
    "type": "markdown"
}


## Generate Curves

In [2]:
def generate_smooth_curve(num_points=100, seed=None, wave_combinations=None):
    """
    Generate a smooth random curved line with multiple wave combinations using a fixed seed.

    Parameters:
        num_points (int): Number of data points to generate.
        seed (int, optional): Random seed for reproducibility.
        wave_combinations(list,optional): List of wave combinations for each frequency component.

    Returns:
    - x: X values.
    - y: Y values.
    """
    if seed is not None:
        np.random.seed(seed)  # Set seed for reproducibility
    
    freq_factors = np.random.randint(1, 3, size=4).tolist()
    amp_factors = np.random.uniform(0.5, 10, size=4).tolist()
    noise_level = np.random.uniform(0, 0.02)
    x_shift = np.random.uniform(0, 3)
    y_shift = np.random.uniform(0, 3)

    if wave_combinations is None:
        wave_combinations = [['sin', 'sinc']]

    x = np.linspace(0, 10, num_points)
    y = np.zeros_like(x)

    for a, f, waves in zip(amp_factors, freq_factors, wave_combinations):
        for w in waves:
            if w == 'sin':
                y += a * np.sin(f * (x + x_shift))
            elif w == 'cos':
                y += a * np.cos(f * (x + x_shift))
            elif w == 'sinc':
                y += a * np.sinc(f * ((x + x_shift)))
            elif w == 'tanh':
                y += a * np.tanh(f * ((x + x_shift)))
            elif w == 'exp':
                y += a * np.exp(-0.5 * f * ((x + x_shift)))

    y += (np.random.normal(scale=noise_level, size=len(x)) + y_shift)
        # Stack into a 2D array
    data = np.column_stack((x, y))
    return x, y

## Plot Altair Vis

We now want to generate the datasets that will go into our vega charts. We don't yet have to worry about rendering these, we'll just define the functions to generate the data.

In [3]:
import random
def generate_random_colors(n, colors=None, seed=None):
    """
    Return a list of n colors.
    If `colors` is provided, it will return the first n colors from that list.
    Otherwise, generate n random hex colors using a reproducible random seed.
    """
    if colors is not None:
        if len(colors) < n:
            raise ValueError(f"Not enough colors provided. Needed {n}, but got {len(colors)}.")
        return colors[:n]
    else:
        rnd = random.Random(seed)
        return [f"#{rnd.randint(0, 0xFFFFFF):06x}" for _ in range(n)]



def plot_altair_curve(seed=None, num_curves=3, opacity=0.3, colors = None):
    """
    Generate and plot multiple smooth random curved lines using Altair with shaded areas.
    Automatically scales the y-axis to fit the minimum and maximum values across all curves.
    """
    if seed is not None:
        np.random.seed(seed)  # Set seed for reproducibility
    
    curves_data = []
    shaded_data = []
    y_min_global = float('inf')
    y_max_global = float('-inf')

    for i in range(num_curves):
        curve_seed = seed + i if seed is not None else None
        x, y = generate_smooth_curve(num_points=100, seed=curve_seed)
        df = pd.DataFrame({'X': x, 'Y': y, 'Curve': f'Curve {i+1}'})
        curves_data.append(df)
        y_min_global = min(y_min_global, np.min(y))
        y_max_global = max(y_max_global, np.max(y))
    
    for df in curves_data:
        df_shade = df.copy()
        df_shade['Y0'] = y_min_global  # Shade from the global minimum y-value up to the curve
        shaded_data.append(df_shade)
    
    all_curves = pd.concat(curves_data)
    all_shaded = pd.concat(shaded_data)
    
    y_scale = alt.Scale(domain=[y_min_global, y_max_global])  # Auto-scale y-axis
    
    # line_chart = alt.Chart(all_curves).mark_line(opacity=opacity).encode(
    #     x='X:Q',
    #     y=alt.Y('Y:Q', scale=y_scale),
    #     color=alt.Color('Curve:N', legend=alt.Legend(title="Curves"))
    # )
    #custom_palette = ['#ff0000', '#00ff00', '#0000ff', '#aaaaaa']
    # Handle colors if not explicitly passed
    if colors is None:
        curve_names = all_shaded['Curve'].unique()
        color_list = generate_random_colors(len(curve_names), 
                                            #colors=custom_palette, 
                                            seed=seed)
        color_scale = dict(zip(curve_names, color_list))
        colors = alt.Color('Curve:N',
                           scale=alt.Scale(domain=list(color_scale.keys()),
                                           range=list(color_scale.values())),
                           legend=None)

    shaded_chart = alt.Chart(all_shaded).mark_area(opacity=opacity).encode(
        x='X:Q',
        y=alt.Y('Y0:Q', scale=y_scale),
        y2='Y:Q',
        color=colors
    )
    
    #return (shaded_chart + line_chart).properties(
    return (shaded_chart).properties(
        width=400,
        height=300,
        title="Curves"
    )

# Use the function to print a test plot
plot_altair_curve(seed=44, num_curves=3, opacity=0.3)

alt.Chart(...)

# Generate Opacity Pairs for Flexible Data Generation 

Added on March 25

In [4]:
def generate_opacity_pairs(base_opacity=0.5, steps_config=[(0.01, 0.02), (0.03, 0.06), (0.1, 0.2)], min_val=0.0, max_val=1.0):
    pairs = set()
    for step_size, max_diff in steps_config:
        num_steps = int(max_diff / step_size)
        for i in range(1, num_steps + 1):
            delta = round(i * step_size, 5)
            lower = round(base_opacity - delta, 5)
            upper = round(base_opacity + delta, 5)
            if min_val <= lower <= max_val:
                pairs.add((base_opacity, lower))
            if min_val <= upper <= max_val:
                pairs.add((base_opacity, upper))
    return [list(pair) for pair in sorted(pairs)]

# Side by Side

For this study, we need to generate pairs of scatterplots and pairs of parallel coordinate plots. We will create two generalized functions which take in two data frames whose columns are 'X' and 'Y' and whose entries are tuples, indicating the coordinates. These functions will each return a vega-altair chart that will be added as components.

In [5]:
import random 

def get_shuffled_opacity(opacityGroup, rnd):
    """
    Shuffle the opacity group using a provided random.Random instance.
    Returns (opacity_left, opacity_right)
    """
    shuffled = rnd.sample(opacityGroup, k=2)
    return shuffled[0], shuffled[1]

#Shuffle a given color list (or pick n from it) using a specific random seed
#def generate_random_colors(n, colors=None, seed=None):
    """
    Shuffle a given color list (or pick n from it) using a specific random seed.
    """
    if colors is None:
        raise ValueError("A list of base colors must be provided.")
    rnd = random.Random(seed)
    return rnd.sample(colors, k=n)


def generate_random_colors(n, colors=None, seed=None):
    """
    Return a list of n colors in the exact order provided in the palette.
    If `colors` is provided, it returns the first n colors as-is (no shuffle).
    """
    if colors is not None:
        if len(colors) < n:
            raise ValueError(f"Not enough colors provided. Needed {n}, but got {len(colors)}.")
        return colors[:n]  # Keep original order
    else:
        rnd = random.Random(seed)
        return [f"#{rnd.randint(0, 0xFFFFFF):06x}" for _ in range(n)]
    

def plot_side_by_side(seed=None, num_curves=3, opacityGroup=None, base_opacity=0.5, shuffle=True):
    """
    Generate and display two Altair charts side by side with different opacities.
    """
    if opacityGroup is None:
        # Use the first pair generated if no specific group is provided
        opacity_pairs = generate_opacity_pairs(base_opacity)
        if not opacity_pairs:
            raise ValueError("No valid opacity pairs generated.")
        opacityGroup = opacity_pairs[0]
    #chart1 = plot_altair_curve(seed=seed, num_curves=num_curves, opacity=opacityGroup[0])
    #chart2 = plot_altair_curve(seed=seed, num_curves=num_curves, opacity=opacityGroup[1])
    #return alt.hconcat(chart1, chart2)

    if shuffle:
        # give shuffle seed based on opacity generated
        shuffleSeed = opacityGroup[0]*10+opacityGroup[1]*10 
        rnd = random.Random(shuffleSeed)
        opacity_left, opacity_right = get_shuffled_opacity(opacityGroup,rnd)
    else:
        shuffled = opacityGroup  # Keep original order
        opacity_left, opacity_right = shuffled

    # Define curve labels
    curve_names = [f'Curve {i+1}' for i in range(num_curves)]


# ================= Here to change the colors [fixed or random] ======================

    # Define fixed color palettes
    custom_palette01 = ['#ffa8d1', '#31b12f', '#0059ff']
    custom_palette02 = ['#ffa8d1', '#31b12f', '#0059ff']

    # Randomly assign color palettes to left and right sides
    swap_palettes = rnd.choice([True, False])
    if swap_palettes:
        palette_left = custom_palette02
        palette_right = custom_palette01
    else:
        palette_left = custom_palette01
        palette_right = custom_palette02

    # Generate reproducible shuffled colors for each side
    left_color_seed = seed if seed is not None else 0
    right_color_seed = seed + 1000 if seed is not None else 1000

    left_colors = generate_random_colors(len(curve_names), 
                                         colors=palette_left
                                         #,seed=left_color_seed
                                         )
    
    right_colors = generate_random_colors(len(curve_names), 
                                          colors=palette_right
                                          #,seed=right_color_seed
                                          )

# =====================================================================================

    # Create Altair color encodings using the generated color lists
    colors_left = alt.Color(
        'Curve:N',
        scale=alt.Scale(domain=curve_names, range=left_colors),
        legend=None
    )

    colors_right = alt.Color(
        'Curve:N',
        scale=alt.Scale(domain=curve_names, range=right_colors),
        legend=None
    )
    print(colors_left)
    print(colors_right)
    # Generate the two charts with different opacities and color encodings
    chart1 = plot_altair_curve(
        seed=seed,
        num_curves=num_curves,
        opacity=opacity_left,
        colors=colors_left
    )

    chart2 = plot_altair_curve(
        seed=seed,
        num_curves=num_curves,
        opacity=opacity_right,
        colors=colors_right
    )


    return alt.hconcat(chart1, chart2).resolve_scale(color='independent')

chart = plot_side_by_side(seed=42, num_curves=3, base_opacity=0.5)
chart

Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})


alt.HConcatChart(...)

# Generate Vega Spec to combine Generated Data and Plots

Now that we have our functions to create the individual chart, we want a function that returns the correct vega spec when given the number of points, the correlation values, and the visualization type ('scatterPlot' or 'parallelPlot').


In [6]:
def create_vega_spec(visType, seed, num_curves=3, opacityGroup=None, base_opacity=0.5):
    """
    Generate a Vega spec from the Altair chart by converting it to Vega-Lite and then to Vega.
    """
    if visType == 'altairPlot':
        if opacityGroup is None:
            # If not provided, pick the first valid pair
            opacity_pairs = generate_opacity_pairs(base_opacity)
            if not opacity_pairs:
                raise ValueError("No valid opacity pairs generated.")
            opacityGroup = opacity_pairs[0]
        chart = plot_side_by_side(seed=seed, num_curves=num_curves, opacityGroup=opacityGroup)
    else:
        raise ValueError("Unsupported visualization type. Use 'altairPlot'.")
    
    vega_lite_spec = chart.to_json()
    vega_spec = vlc.vegalite_to_vega(vega_lite_spec, vl_version="5.20")

    vega_spec["autosize"] = {
        "type": "fit-y",
        "resize": "true",
        "contains": "content"
    }

    vega_spec["padding"] = {
        "left": 150,
        "right": 1000,
        "top": 20,
        "bottom": 20
    }

    # Centering charts
    # vega_spec["usermeta"] = vega_spec.get("usermeta", {})
    # vega_spec["usermeta"]["alignment"] = "center"
    return vega_spec
# We can print the test vega specification above to inspect its contents.
my_vega_spec = create_vega_spec(visType='altairPlot', seed=42, num_curves=3, opacityGroup=None)
#print(my_vega_spec)
json_spec = json.dumps(my_vega_spec, indent=2)
print(json_spec)


Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
{
  "$schema": "https://vega.github.io/schema/vega/v5.json",
  "background": "white",
  "padding": {
    "left": 150,
    "right": 1000,
    "top": 20,
    "bottom": 20
  },
  "height": 300,
  "data": [
    {
      "name": "data-8722fe2a6e5ca3148ad8c65dae86672c",
      "format": {},
      "values": [
        {
          "Curve": "Curve 1",
          "X": 0,
          "Y": 6.524268345675902,
          "Y0": -6.072722065208106
        },
        {
          "Curve": "Curve 1",
          "X": 0.101010101010101,
          "Y": 5.703314364269875,
          "Y0": -6.072722065208106
        },
        {
          "Curve": "Curve 1",
          "X": 0.202020202020202,
  

# Creating The Component Function & Interaction Signals for ReVISit Trials

The `component_function` is used to transform every component in a given sequence to any new component. If we have a sequence that is the correct _structure_, then we call the `component()` method on that sequence and pass in the desired `component_function`. Any `meta` attributes in the original components are passed in as arguments to the `component_function`. 

We'll create a component function which takes in the visualization type, the correlation values, and the number of points and returns the correct vega specification component.

In [7]:
#def xor_cipher(data: str, key: str) -> str:
#    """Encrypt or decrypt data using XOR and a repeating key."""
#    return ''.join(chr(ord(c) ^ ord(key[i % len(key)])) for i, c in enumerate(data))
# Researcher key
#secret_key = "revisitStudy" 
def component_function(seed=None, opacityGroup=None, base_opacity=0.5):
    if seed is not None:
        if opacityGroup is None:
            # generate opacity pairs and select the first one
            opacity_pairs = generate_opacity_pairs(base_opacity)
            if not opacity_pairs:
                raise ValueError("No valid opacity pairs generated.")
            opacityGroup = opacity_pairs[0]
        
        # assign a not randomized shuffle seed
        shuffleSeed = opacityGroup[0]*10 + opacityGroup[1]*10 
        rnd = random.Random(shuffleSeed)
        leftImage,rightImage = get_shuffled_opacity(opacityGroup,rnd)
        # calculate which one is the correct answer
        correct_answer = "Left Image" if leftImage > rightImage else "Right Image"

        vega_response=rvt.response(
            id='button_selected',
            prompt='Please select the figure with higher opacity level (higher opacity means less transparent):',
            type='buttons',
            options=["Left Image", "Right Image"]
        )

        correct_answer_obj = rvt.answer(
            id='button_selected',
            answer=correct_answer
        )

        metadata_response = rvt.response(
            id='seed_L_R',
            prompt="seed_L_R (hidden)",
            type='shortText',
            hidden=True,
            required=False
        )

        metadata_correct_answer ={
                'seed': seed,
                'imageShuffleSeed':shuffleSeed,
                'opacity_left': leftImage,
                'opacity_right': rightImage
            }

        metadata_answer_obj = rvt.answer(
            id='seed_L_R',
            answer=metadata_correct_answer
        )

        vega_spec = create_vega_spec(
            visType='altairPlot',
            seed=seed,
            num_curves=3,
            opacityGroup=opacityGroup
        )

        # Create 'config' if missing
        if 'config' not in vega_spec:
            vega_spec['config'] = {}
        
        return rvt.component(
            type='vega',
            config=vega_spec,
            # alignment="center", # Centering Vega Items
            #component_name__=xor_cipher(f'{seed}-{opacityGroup[0]},{opacityGroup[1]}',secret_key),
            component_name__=f'{seed}-{leftImage},{rightImage}',
            response=[vega_response,metadata_response],
            correctAnswer=[correct_answer_obj, metadata_answer_obj]
        )
    
# You can print the output of our component function with some test values.
comp_func = component_function(seed=42)
print(comp_func)
#print(component_function(seed=42, opacityGroup=[0.3, 0.6]))



Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
{
    "config": {
        "$schema": "https://vega.github.io/schema/vega/v5.json",
        "background": "white",
        "padding": {
            "left": 150,
            "right": 1000,
            "top": 20,
            "bottom": 20
        },
        "height": 300,
        "data": [
            {
                "name": "data-8722fe2a6e5ca3148ad8c65dae86672c",
                "format": {},
                "values": [
                    {
                        "Curve": "Curve 1",
                        "X": 0,
                        "Y": 6.524268345675902,
                        "Y0": -6.072722065208106
                    },
                    {
      

# Creating The Consent Form Component

Added on April 26

In [8]:
def component_consentform():
    
    consent01_response=rvt.response(
        id='signature',
        location='belowStimulus',
        prompt='Your signature',
        # secondaryText='I have read the consent form and recognize that my participation in this study is entirely voluntary and that I am free to withdraw at any time during the course of the study without consequence. I understand that any information resulting from this study will be strictly confidential. I realize that I may ask for further information about this study if I wish to do so at any time. I agree to participate in this study.',
        type='shortText',
        placeholder='Enter your name as your consent form signature',
        required=True
    )

    consent02_response=rvt.response(
        id='accept',
        requiredValue='Accept',
        location='belowStimulus',
        prompt='Do you consent to the study and wish to continue?',
        required=True,
        type='radio',
        options=['Decline',
                 'Accept']
    )

    #metadata_response = rvt.response(
    #    id='seed_L_R',
    #    prompt="seed_L_R (hidden)",
    #    type='shortText',
    #    hidden=True,
    #    required=False
    #)

        
    return rvt.component(
        component_name__='Consent Form',
        type='markdown',
        path='./assets/consent.md',
        response=[consent01_response,consent02_response]
    )
    
# You can print the output of our component function with some test values.
comp_consent = component_consentform()
print(comp_consent)

{
    "correctAnswer": [],
    "path": "./assets/consent.md",
    "response": [
        {
            "id": "signature",
            "location": "belowStimulus",
            "placeholder": "Enter your name as your consent form signature",
            "prompt": "Your signature",
            "required": true,
            "type": "shortText"
        },
        {
            "id": "accept",
            "location": "belowStimulus",
            "options": [
                "Decline",
                "Accept"
            ],
            "prompt": "Do you consent to the study and wish to continue?",
            "required": true,
            "requiredValue": "Accept",
            "type": "radio"
        }
    ],
    "type": "markdown"
}


# Creating The Demographics Info Component

Added on April 26

In [9]:
def component_demographics():
    
    demographics01_response=rvt.response(
        id='genderidentity',
        location='belowStimulus',
        prompt='What is your gender identity?',
        secondaryText='Multiple-selections allowed',
        type='checkbox',
        options=['Woman',
                 'Man',
                 'Non-binary / Genderqueer / Third gender',
                 'Genderfluid / Gender non-conforming',
                 'Agender',
                 'Transgender',
                 'Prefer not to say'],
        withOther=True,
        required=True
    )

    demographics02_response=rvt.response(
        id='agerange',
        location='belowStimulus',
        prompt='What is your age?',
        required=True,
        type='radio',
        options=['Under 18 years',
                 '18-24 years',
                 '25-34 years',
                 '35-44 years',
                 '45-54 years',
                 '55-64 years',
                 '65 years or older',
                 'Prefer not to say']
    )

    demographics03_response=rvt.response(
        id='race',
        location='belowStimulus',
        prompt='What is your racial identity?',
        secondaryText='Multiple-selections allowed',
        required=True,
        withOther=True,
        type='checkbox',
        options=['White',
                 'Hispanic or Latino',
                 'Black or African American',
                 'Asian',
                 'American Indian & Alaskan Native',
                 'Native Hawaiian & Other Pacific Islander',
                 'Multiracial',
                 'Prefer not to say']
    )

    demographics04_response=rvt.response(
        id='education',
        location='belowStimulus',
        prompt='What is the highest degree or level of education you have completed?',
        required=True,
        withOther=True,
        type='radio',
        options=['Less than high school',
                 'High school diploma or equivalent',
                 'Bachelor degree or equivalent',
                 'Master degree or equivalent',
                 'Doctoral degree or equivalent',
                 'Prefer not to say']
    )

    #metadata_response = rvt.response(
    #    id='seed_L_R',
    #    prompt="seed_L_R (hidden)",
    #    type='shortText',
    #    hidden=True,
    #    required=False
    #)

        
    return rvt.component(
        component_name__='Demographics Information',
        type='questionnaire',
        response=[demographics01_response,demographics02_response,demographics03_response,demographics04_response]
    )
    
# You can print the output of our component function with some test values.
comp_demographics = component_demographics()
print(comp_demographics)

{
    "correctAnswer": [],
    "response": [
        {
            "id": "genderidentity",
            "location": "belowStimulus",
            "options": [
                "Woman",
                "Man",
                "Non-binary / Genderqueer / Third gender",
                "Genderfluid / Gender non-conforming",
                "Agender",
                "Transgender",
                "Prefer not to say"
            ],
            "prompt": "What is your gender identity?",
            "required": true,
            "secondaryText": "Multiple-selections allowed",
            "type": "checkbox",
            "withOther": true
        },
        {
            "id": "agerange",
            "location": "belowStimulus",
            "options": [
                "Under 18 years",
                "18-24 years",
                "25-34 years",
                "35-44 years",
                "45-54 years",
                "55-64 years",
                "65 years or older",
                "Prefe

# Creating The Color-Blindness Test Component(s)

Added on April 26

In [10]:
def component_colortest0():
    
    return rvt.component(
        component_name__='Color-Blindness-Test Intro',
        type='markdown',
        path='./assets/color-blindness.md'
    )
    
# You can print the output of our component function with some test values.
comp_colortest0 = component_colortest0()

def component_colortest1():
    
    colortest1_response=rvt.response(
        id='test1-plate2-answer8',
        location='belowStimulus',
        prompt='Enter the number you see:',
        type='numerical',
        withDontKnow=True,
        required=True
    )

    correct_answer_ct1 = rvt.answer(
        id='test1-plate2-answer8',
        answer=8
    )

    return rvt.component(
        component_name__='Color-Blindness-Test 01',
        instruction='What number do you see in this image?',
        type='image',
        path='./assets/Ishihara2.png',
        style={"width": "300px"},
        response=[colortest1_response],
        correctAnswer=[correct_answer_ct1]
    )
    
# You can print the output of our component function with some test values.
comp_colortest1 = component_colortest1()



def component_colortest2():
    
    colortest2_response=rvt.response(
        id='test2-plate8-answer6',
        location='belowStimulus',
        prompt='Enter the number you see:',
        type='numerical',
        withDontKnow=True,
        required=True
    )

    correct_answer_ct2 = rvt.answer(
        id='test2-plate8-answer6',
        answer=6
    )

    return rvt.component(
        component_name__='Color-Blindness-Test 02',
        instruction='What number do you see in this image?',
        type='image',
        path='./assets/Ishihara8.png',
        style={"width": "300px"},
        response=[colortest2_response],
        correctAnswer=[correct_answer_ct2]
    )
    
comp_colortest2 = component_colortest2()



def component_colortest3():
    
    colortest3_response=rvt.response(
        id='test3-plate9-answer45',
        location='belowStimulus',
        prompt='Enter the number you see:',
        type='numerical',
        withDontKnow=True,
        required=True
    )

    correct_answer_ct3 = rvt.answer(
        id='test3-plate9-answer45',
        answer=45
    )

    return rvt.component(
        component_name__='Color-Blindness-Test 03',
        instruction='What number do you see in this image?',
        type='image',
        path='./assets/Ishihara9.png',
        style={"width": "300px"},
        response=[colortest3_response],
        correctAnswer=[correct_answer_ct3]
    )
    
comp_colortest3 = component_colortest3()

print(comp_colortest0,comp_colortest1,comp_colortest2,comp_colortest3)

{
    "correctAnswer": [],
    "path": "./assets/color-blindness.md",
    "response": [],
    "type": "markdown"
} {
    "correctAnswer": [
        {
            "answer": 8,
            "id": "test1-plate2-answer8"
        }
    ],
    "instruction": "What number do you see in this image?",
    "path": "./assets/Ishihara2.png",
    "response": [
        {
            "id": "test1-plate2-answer8",
            "location": "belowStimulus",
            "prompt": "Enter the number you see:",
            "required": true,
            "type": "numerical",
            "withDontKnow": true
        }
    ],
    "style": {
        "width": "300px"
    },
    "type": "image"
} {
    "correctAnswer": [
        {
            "answer": 6,
            "id": "test2-plate8-answer6"
        }
    ],
    "instruction": "What number do you see in this image?",
    "path": "./assets/Ishihara8.png",
    "response": [
        {
            "id": "test2-plate8-answer6",
            "location": "belowStimulus"

# Creating The Intro Page for Opacity Test Components

Added on April 26

In [11]:
def component_opacityintro():
        
    return rvt.component(
        component_name__='Opacity Test Intro Page',
        type='markdown',
        path='./assets/opacityintro.md'
    )
    
# You can print the output of our component function with some test values.
comp_opacityintro = component_opacityintro()
print(comp_opacityintro)

{
    "correctAnswer": [],
    "path": "./assets/opacityintro.md",
    "response": [],
    "type": "markdown"
}


# Permuting the Final Sequence 

Here we generate the different combinations of the correlation values that we'd like. Then, we generate a sequence and being the permutations over our factors. We first permute over the visualization type, then over the number of points, then over all possible correlation value pairs.

When we permute over these factors, the corresponding factored will be added to the `meta` attributes of each component. Before calling the `component` method, these are all "filler" or "placeholder" components with no real value aside from their metadata attributes. Once we call the `component` method, each component is passed through the inputted `component_function` which will take the existing metadata as arguments. Thus, by the end of this method chaining, each component will be the correct vega component.

In [12]:

# Set up multiple base_opacity value (can be changed later if needed)
base_opacities = [0.3, 0.5, 0.7, 0.9]

# Generate all combinations of pairs under base_opacity
dataSet = []
for base in base_opacities:
    pairs = generate_opacity_pairs(base)
    for pair in pairs:
        dataSet.append({'opacityGroup': pair, 'base_opacity': base})

random.shuffle(dataSet) 

factors = []
for i, data in enumerate(dataSet):
    group_index = i // 5
    data['seed'] = 101 + group_index
    factors.append(data)

# Generate all combinations of two values between 1 and 10
#combinations = itertools.combinations(range(1, 11), 2)

# Create the dataset with values divided by 10
#dataSet = [{'opacityGroup': [x / 10, y / 10]} for x, y in combinations]

main_sequence = rvt.sequence(order='fixed')

#main_sequence.permute(
#        factors=[{'seed': 42}],
#        order='latinSquare',
#    ).permute(
#        factors=dataSet,
#        order='random',
#    ).component(component_function)

main_sequence.permute(
    factors=factors,
    order='random'
).component(component_function)
    
sequence = rvt.sequence(order='fixed',components=[introduction,comp_consent,comp_demographics]) + comp_colortest0 + rvt.sequence(order='fixed',components=[comp_colortest1,comp_colortest2,comp_colortest3]) + comp_opacityintro + main_sequence

study = rvt.studyConfig(
    schema='https://raw.githubusercontent.com/revisit-studies/study/v2.1.0/src/parser/StudyConfigSchema.json',
    uiConfig=ui_config,
    studyMetadata=study_metadata,
    sequence=sequence
)

# Prints the entire configuration file which is approximately 150,000 lines of JSON
print(study)


Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  legend: None,
  scale: Scale({
    domain: ['Curve 1', 'Curve 2', 'Curve 3'],
    range: ['#ffa8d1', '#31b12f', '#0059ff']
  }),
  shorthand: 'Curve:N'
})
Color({
  

In [13]:
# Ensure the final study is passed to JSON and the widget
final_study = study

# turn final_study object into JSON and save
final_study_json = final_study.__str__()  # get JSON string
final_study_data = json.loads(final_study_json)  # turn into Python dictionary
#final_study_data["final_studyMetadata"]["title"] = "Opacity Judgment Study"

# save as config.json
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(final_study_data, f, indent=2)

print("✅ config.json generated!")

✅ config.json generated!


# Using `revisitpy_server` to Prepare Our Widget

The `revisitpy` package provides a widget in order to preview our study directly in a Jupyter notebook. We can interact with the study, check that vega signals work, and even create some introductory data ourselves. In order for the widget to work, a local copy of the reVISit must be running on your local computer. If you already have reVISit locally (colloqioully our `study` repo), then all you need to do is navigate to your repository and run `yarn serve`. After this, the widget we create in this jupyter notebook will be useable.

A simpler way to achieve the same goal, however is using the `revisitpy_server` Python package. This is a simple python package which already has the most recent reVISit repository built and runs a server locally. After installing `revisitpy_server`, all that is required is the following:

In [14]:
process = rs.serve()

Server is running in the background at http://localhost:8080


# The Widget

Now that your server is running, we create the widget with the configuration file we created above. When calling the widget, we are assuming that the assets referenced in the configuration file are relative to this notebook. The widget then copies these static assets to the appropriate directory. Since we're currently using the `revisitpy_server` package, you'll see that they copied into the assets of the local virtual environment `revisitpy_server` package.

In [15]:
w = rvt.widget(study, server=True)

# In your own Jupyter notebook, calling `w` will now display the widget in a fully interactive manner.
w

Copying file from ./assets/introduction.md to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/introduction.md
Copying file from ./assets/consent.md to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/consent.md
Copying file from ./assets/color-blindness.md to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/color-blindness.md
Copying file from ./assets/Ishihara2.png to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/Ishihara2.png
Copying file from ./assets/Ishihara8.png to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/Ishihara8.png
Copying file from ./assets/Ishihara9.png to d:\

Widget(config={'$schema': 'https://raw.githubusercontent.com/revisit-studies/study/v2.1.0/src/parser/StudyConf…

# Optional: Data Collection

Now that we have the widget running, we can check out some sample data that would be generated from a user. Start by going through a small portion of the study. Once you've gone through the desired number of components inside the widget, navigate to the analysis dashboard using the 'Analysis' tab in the upper left-hand corner. Here you'll see individual participants and the data that they've generated. 

From here, we can export this data back into our Jupyter notebook. Start by clicking the "Download as Tidy CSV" on the right-hand side above the table. Here you'll be shown a preview of the CSV file with some additional options to truncate the data. In the bottom right-hand corner, you'll see a button with the Python icon. Clicking on this button will send the Tidy CSV back to the Jupyter notebook. Once the button is clicked, we can preview the data like so:

In [16]:
w.get_df()

KeyError: 'rows'

# Optional: Terminate the server

Closing the notebook will automatically terminate the server. If you'd rather do this manually, you can do the following.

In [17]:
process.terminate()